In [ ]:
import streamlit as st
import pdfplumber
import openai  # Requires OpenAI API key

# Set OpenAI API Key
openai.api_key = st.secrets["OPENAI_API_KEY"]  # Ensure this is set in Streamlit secrets

def extract_text_from_pdf(uploaded_file):
    """Extract text from uploaded PDF resume."""
    with pdfplumber.open(uploaded_file) as pdf:
        text = "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
    return text

def analyze_resume_with_gpt(resume_text, job_description):
    """Use GPT-4 to analyze resume relevance to job description."""
    prompt = f"""
    You are an AI assistant that evaluates resumes against job descriptions. 
    The user has provided their resume text and a job description.
    
    **Resume:** 
    {resume_text}
    
    **Job Description:** 
    {job_description}
    
    Provide a structured analysis with:
    1. **Relevance Score** (0-100%) based on keyword and skill match.
    2. **Missing Keywords/Skills** that should be included.
    3. **Grammar & Style Suggestions** for professional improvement.
    4. **Overall Summary** of how well the resume aligns with the job.
    """
    
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "system", "content": prompt}]
    )
    
    return response["choices"][0]["message"]["content"]

# Streamlit UI
st.title("📄 AI-Powered Resume Critique Tool (GPT-4 Enhanced)")
st.write("Upload your resume and paste the job description below to get AI-driven feedback.")

# Resume upload
uploaded_resume = st.file_uploader("📂 Upload Your Resume (PDF only)", type=["pdf"])

# Job description input
job_description = st.text_area("📝 Paste the Job Description Here:", height=200)

if st.button("Analyze Resume"):
    if uploaded_resume and job_description:
        resume_text = extract_text_from_pdf(uploaded_resume)
        if resume_text:
            feedback = analyze_resume_with_gpt(resume_text, job_description)
            st.subheader("🔍 AI Resume Analysis Result:")
            st.write(feedback)
        else:
            st.error("❌ Could not extract text from the PDF. Please upload a valid resume.")
    else:
        st.warning("⚠️ Please upload a resume and paste the job description.")
